# Materialized Inference + Webhooks (MLServe.com)

This notebook demonstrates materialized inference in MLServe.com: instead of returning predictions directly to the caller, MLServe.com can store predictions keyed by an `entity_id` (e.g. user/customer id). A downstream service can later retrieve the prediction using only the identifier—no feature preprocessing or joins in the request path.

We also demonstrate an MVP webhook notification mechanism: when a prediction is materialized, MLServe sends an HTTP POST event to a customer-owned endpoint, allowing a backend to react immediately (e.g., fetch the prediction and update its cache/DB for the frontend).

## What we’ll build end-to-end

1. Start a local webhook receiver (`localhost:8001`) to observe events
2. Expose it publicly via `ngrok` so MLServe.com can reach it
3. Train and deploy a simple churn model on MLServe.com
4. Register the webhook URL in MLServe.com
5. Run materialized inference (store predictions by entity_id)
6. Fetch the stored prediction using `fetch_materialized`

## Step 0 — Prerequisites

Before running the notebook:

* You need MLServe.com credentials (stored as environment variables)
* You need `ngrok` installed locally
* You need two terminals:
    * Terminal A: webhook receiver server
    * Terminal B: ngrok tunnel
* Environment variables expected:
    * `USERNAME` (MLServe.com username/email)
    * `TOKEN` (your MLServe.com password or API token used by `client.login`)

## Step 1 — Start a local webhook receiver (localhost:8001)

We will run a tiny FastAPI app that prints incoming webhook payloads to the terminal.
This simulates a customer backend receiving “prediction is ready” events.

Create a file called `receiver.py` with the following content.

```python
from fastapi import FastAPI, Request
import uvicorn

app = FastAPI()

@app.post("/mlserve/webhook")
async def mlserve_webhook(req: Request):
    body = await req.body()
    print("\n=== WEBHOOK RECEIVED ===")
    print("Headers:", dict(req.headers))
    print("Body:", body.decode("utf-8"))
    return {"ok": True}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8001)
```

## Step 1A — Run the receiver (Terminal A)

Start the receiver server in a separate terminal:

```sh
python receiver.py
```

You should see a log line indicating the server is listening on port 8001.

## Step 2 — Expose localhost:8001 using ngrok (Terminal B)

MLServe.com runs remotely, so it can’t call `localhost`. We use `ngrok` to expose our local receiver via a public HTTPS URL.

In a different terminal:

```sh
ngrok http 8001

```

ngrok will print a public forwarding URL like:

```sh
https://<something>.ngrok-free.app  ->  http://localhost:8001
```

Copy the HTTPS URL and append the path:
`/mlserve/webhook`

Example:
`https://abc123.ngrok-free.app/mlserve/webhook`

We’ll use this URL when we register the webhook in MLServe.com.

## Step 3 — Train a demo churn model locally

We’ll generate a small synthetic churn dataset, train an XGBoost classifier, and deploy it to MLServe.com.

This model is purely for demonstration—what matters is the serving pattern: **materialize + fetch + webhook.**

In [14]:
from sklearn.datasets import load_iris
import xgboost
from xgboost import XGBClassifier
from mlserve_sdk.client import MLServeClient
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()

def generate_churn_data(n_samples=1000, missing_frac=0.05, random_state=42):
    np.random.seed(random_state)

    data = {
        "customer_id": np.arange(1, n_samples+1),
        "age": np.random.randint(18, 80, n_samples),
        "tenure_months": np.random.randint(1, 72, n_samples),
        "monthly_charges": np.round(np.random.uniform(20, 120, n_samples), 2),
        "total_charges": np.round(np.random.uniform(20, 8000, n_samples), 2),
        "contract_type": np.random.choice(
            ["Month-to-month", "One year", "Two year"], n_samples, p=[0.6, 0.25, 0.15]
        ),
        "payment_method": np.random.choice(
            ["Electronic check", "Mailed check", "Bank transfer", "Credit card"], n_samples
        ),
        "internet_service": np.random.choice(
            ["DSL", "Fiber optic", "No"], n_samples, p=[0.3, 0.5, 0.2]
        ),
        "gender": np.random.choice(["Male", "Female"], n_samples),
        "has_phone_service": np.random.choice(["Yes", "No"], n_samples, p=[0.9, 0.1]),
        "num_dependents": np.random.poisson(1, n_samples),
    }

    X = pd.DataFrame(data)

    # Inject missing values
    if missing_frac > 0:
        for col in X.columns.drop("customer_id"):
            X.loc[X.sample(frac=missing_frac, random_state=random_state).index, col] = np.nan

    # Synthetic churn probability
    prob_churn = (
        0.3 * (X["contract_type"] == "Month-to-month").astype(float) +
        0.25 * (X["internet_service"] == "Fiber optic").astype(float) +
        0.15 * (X["payment_method"] == "Electronic check").astype(float) +
        0.002 * (X["monthly_charges"].fillna(60)) +
        0.01 * (X["num_dependents"].fillna(0) == 0).astype(float) +
        np.random.normal(0, 0.1, n_samples)
    )
    prob_churn = 1 / (1 + np.exp(-prob_churn))

    y = pd.Series(np.random.binomial(1, prob_churn), name="churn")
    return X, y

# Generate data
X, y = generate_churn_data(n_samples=1000, missing_frac=0.05)

# Drop id column (we'll use DataFrame index as entity_id later)
X.drop(columns=["customer_id"], inplace=True)

# Set categoricals for XGBoost
for col in ["contract_type", "payment_method", "internet_service", "gender", "has_phone_service"]:
    X[col] = X[col].astype("category")

# Train model
model = XGBClassifier(enable_categorical=True, tree_method="hist")
model.fit(X, y)

print("Local model trained. Accuracy:", model.score(X, y))

## Step 4 — Authenticate and deploy the model to MLServe.com

We:

1. authenticate using environment variables
2. determine the next version for model `churn`
3. deploy the model
4. wait for the deployment to finish

In [8]:
USERNAME = os.getenv("USERNAME")
TOKEN = os.getenv("TOKEN")

client = MLServeClient()
client.login(USERNAME, TOKEN)

# Pick next version (fallback to v1)
try:
    lv = client.get_latest_version("churn")
    next_version = lv["next_version"]
except Exception:
    next_version = "v1"

print("Deploying churn model version:", next_version)

resp = client.deploy(
    model=model,
    name="churn",
    version=next_version,
    features=list(X.columns),
    background_df=X.sample(100),
    metrics={"accuracy": float(model.score(X, y))},
    task_type="classification"
)

print("✅ Deployment requested:")
print(resp)

deployment_id = resp["deployment_id"]
print("🚀 Deployment started:", deployment_id)

final_status = client.wait_for_deployment(deployment_id)
print("✅ Final deployment status:", final_status)

## Step 5 — Register webhook (so MLServe.com can notify us)

Now we tell MLServe.com where to send events when a prediction is materialized.

Use the **public ngrok URL** you copied earlier (ending with `/mlserve/webhook`).

When materialization happens, you should see the receiver terminal print:
`=== WEBHOOK RECEIVED ===`

In [13]:
# Replace with your ngrok URL
WEBHOOK_URL = "https://brandy-transthoracic-adrien.ngrok-free.dev/mlserve/webhook"

client.set_webhook(
    url=WEBHOOK_URL,
    secret="test_secret",
    is_active=True
)

print("✅ Webhook registered:", WEBHOOK_URL)

## Step 6 — Materialize predictions

Instead of returning predictions directly, we call predict with `materialize=True`.

Key points:

* `inputs` contains feature rows
* `entity_ids` contains stable identifiers for each row
* MLServe.com stores the results keyed by `(model, entity_id)`
* The response is an ACK containing storage keys
* For this demo we use the DataFrame index as `entity_id`.

In [15]:
TEST_DATA = {
    "features": X.head(10).columns.tolist(),
    "inputs": X.head(10).values.tolist(),
    "entity_ids": X.head(10).index.astype(str).tolist(),
    "materialize": True
}

ack = client.predict("churn", next_version, TEST_DATA, materialize=True)
print("ACK:", ack)
print("\nStored keys:", ack.get("keys"))

What to observe now:

* In the notebook: you get materialized=True and keys
* In Terminal A (receiver): you should see `=== WEBHOOK RECEIVED ===`

## Step 7 — Fetch prediction later by entity_id only

Now we retrieve a prediction using only:
* model name (`churn`)
* entity_id (`"0"` for the first row)

No preprocessing, no feature lookup in the backend.

In [16]:
pred = client.fetch_materialized(
    name="churn",
    entity_id="0",
    max_age_seconds=300
)

print("Fetched materialized prediction:")
print(pred)

## Wrap-up: What this enables in real systems

This pattern supports a common production architecture:
* Data pipeline triggers inference and materialization (near real-time or batch)
* MLServe.com stores predictions keyed by entity_id
* MLServe.com notifies the backend via webhook that the prediction is ready
* Backend fetches and persists/caches the result
* Frontend reads from backend without running feature preprocessing or calling the model directly